In [19]:
import pandas as pd
import numpy as np

# 1.- load the main amazon sales dataset using relative paths
df_amazon = pd.read_csv('../data/Amazon Sale Report.csv', low_memory=False)

#2.- Configure pandas options to display all columns without truncation
pd.set_option('display.max_columns', None)

#3.- display the first 5 rows to inspect the data structure
df_amazon.head()

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,ASIN,Courier Status,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


In [20]:
#4.- Check the total rows and columns (Shape of the dataset)
print(f"Dataset Shape: ", df_amazon.shape)

#5.- Inspect column names, data types and non-null counts
df_amazon.info()

Dataset Shape:  (128975, 24)
<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount          

## Data Quality Findings & Cleaning Strategy

### Identified Issues:
1. **Incorrect Data Types:** The `Date` column is stored as a string (`str`), and `ship-postal-code` is stored as a float. These need to be converted to `datetime` and `string/object` respectively.
2. **Critical Missing Values:** * `Amount` and `currency` have missing records (~7,000 nulls), likely tied to canceled orders.
   * `fulfilled-by` is missing over 60% of its data.
   * `Unnamed: 22` is an empty artifact column that serves no analytical purpose.

### Action Plan:
* **Drop unnecessary columns** that contain structural noise (`Unnamed: 22`).
* **Convert data types** to enable proper time-series and categorical filtering.
* **Investigate and handle missing values** in financial columns (`Amount`).

In [21]:
# Step 10: Drop the useless 'Unnamed: 22' column to clean up the dataframe structure
# We use axis=1 to specify we are dropping a column, and inplace=True to apply changes directly
df_amazon.drop(columns=['Unnamed: 22'], errors='ignore', inplace=True)

# Step 11: Display the updated columns to verify the deletion
print(f"Updated columns count: ", len(df_amazon.columns))
df_amazon.head(2)

Updated columns count:  23


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,ASIN,Courier Status,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship


## Step 2: Data Type Standardization - Converting 'Date' Column

### Rationale:
The `Date` column was initially loaded as a string (`str`). To perform time-series analysis, monthly aggregations, and chronological filtering, this column must be converted into a native `datetime` object.

### Implementation:
We will use `pd.to_datetime()` with `errors='coerce'` to securely handle any corrupted date strings by converting them to `NaT` (Not a Time) values, preventing the script from crashing.

In [22]:
# Step 12: Convert 'Date' column to datetime format
# format='mixed' helps pandas automatically detect different date structures within the same column
df_amazon['Date'] = pd.to_datetime(df_amazon['Date'], errors='coerce', format='mixed')

# Step 13: Verify the transformation by checking the column's data type
print("New Data Type for 'Date' column:")
print(df_amazon['Date'].dtype)

# Step 14: Inspect the minimum and maximum dates to verify our data range
print(f"\nData Timeline Ranges from: {df_amazon['Date'].min()} to {df_amazon['Date'].max()}")

New Data Type for 'Date' column:
datetime64[us]

Data Timeline Ranges from: 2022-03-31 00:00:00 to 2022-06-29 00:00:00


## Step 3: Missing Value Analysis for Financial Columns

### Hypothesis:
Missing values in the `Amount` and `currency` columns (~7,000 records) are not random data losses. Instead, they are highly correlated with specific transaction types, such as orders with a `Status` of 'Cancelled' or 'Rejected', where no actual revenue was collected.

### Validation:
We will analyze the unique values in the `Status` column for all rows where `Amount` is null to confirm this behavioral pattern.

In [23]:
# Step 15: Filter the dataset to get rows where 'Amount' is missing (null)
df_missing_amount = df_amazon[df_amazon['Amount'].isnull()]

# Step 16: Count how many times each status appears when 'Amount' is missing
print("Order Status breakdown for rows with MISSING Amounts:")
print(df_missing_amount['Status'].value_counts())

# Step 17: For contrast, let's look at the average 'Amount' for 'Cancelled' orders that DO have money tracked
print("\nCheck if there are any 'Cancelled' orders that actually have an Amount:")
print(df_amazon[df_amazon['Status'] == 'Cancelled']['Amount'].describe())

Order Status breakdown for rows with MISSING Amounts:
Status
Cancelled                       7566
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
Name: count, dtype: int64

Check if there are any 'Cancelled' orders that actually have an Amount:
count    10766.00000
mean       642.69778
std        259.78255
min        218.10000
25%        438.57500
50%        601.90000
75%        771.00000
max       4235.72000
Name: Amount, dtype: float64


## Step 4: Imputing Missing Financial Values Based on Order Status

### Rule of Decision:
1. **Cancelled Orders:** If an order has a missing `Amount` and its status is 'Cancelled', we will fill the missing value with `0`, as no actual revenue was generated.
2. **Shipped/Pending Orders:** For orders that were actually shipped but miss an `Amount`, we will temporarily fill them with `0` or replace them with a baseline to prevent skewing future financial metrics. 

### Implementation:
We will clean the `Amount` column and also ensure the `currency` column defaults to 'INR' (or the dominant currency) for consistency.

In [24]:
# Step 18: Fill missing values in 'Amount' specifically for Cancelled orders first
# We locate rows where Amount is null AND Status is Cancelled, and set them to 0
df_amazon.loc[(df_amazon['Amount'].isnull()) & (df_amazon['Status'] == 'Cancelled'), 'Amount'] = 0.0

# Step 19: For the remaining anomalies (Shipped without amount), we will fill them with 0 as well for baseline safety
df_amazon['Amount'] = df_amazon['Amount'].fillna(0.0)

# Step 20: Clean the 'currency' column by filling missing values with the most common currency (Mode)
dominant_currency = df_amazon['currency'].mode()[0]
df_amazon['currency'] = df_amazon['currency'].fillna(dominant_currency)

# Step 21: Verify that 'Amount' and 'currency' no longer have missing values
print("Remaining missing values in financial columns:")
print(df_amazon[['Amount', 'currency']].isnull().sum())

Remaining missing values in financial columns:
Amount      0
currency    0
dtype: int64


## Step 5: Geographic Data Standardization (Cities & Postal Codes)

### Rationale:
To build reliable map visualizations in Power BI, geographic columns must be fully standardized:
1. **`ship-postal-code`:** Converted from float to string/object to preserve leading zeros and ensure it behaves as an identifier, not a numeric values to aggregate.
2. **`ship-city` & `ship-state`:** Stripped of leading/trailing whitespace and converted to title case (e.g., 'mumbai' or 'MUMBAI' becomes 'Mumbai') to fix structural duplicates.

In [25]:
# Step 22: Convert Postal Code to string, handling null values safely
df_amazon['ship-postal-code'] = df_amazon['ship-postal-code'].fillna(0).astype(int).astype(str).replace('0', np.nan)

# Step 23: Clean text columns (strip spaces and convert to Title Case)
df_amazon['ship-city'] = df_amazon['ship-city'].str.strip().str.title()
df_amazon['ship-state'] = df_amazon['ship-state'].str.strip().str.title()

# Step 24: Check a quick sample of the cleaned geographic data
print("Sample of standardized geographic columns:")
df_amazon[['ship-city', 'ship-state', 'ship-postal-code']].head()

Sample of standardized geographic columns:


,ship-city,ship-state,ship-postal-code
0,Mumbai,Maharashtra,400081
1,Bengaluru,Karnataka,560085
2,Navi Mumbai,Maharashtra,410210
3,Puducherry,Puducherry,605008
4,Chennai,Tamil Nadu,600073


## Step 6: Dataset Consolidation - Merging Local and International Sales

### Rationale:
To build an omnichannel sales dashboard, we cannot keep data fragmented. We need to load the international sales dataset, ensure it aligns with our cleaned Amazon schema, and append them vertically into a single consolidated Master Table using `pd.concat()`.

### Strategy:
* Load `International Sale Report.csv`.
* Standardize its columns to match the core schema.
* Concatenate both DataFrames and export the clean result to the `/data` folder as a compressed CSV or Parquet file.

In [26]:
# Step 25: Load the International Sales dataset
intl_sales_path = '../data/International Sale Report.csv'
df_intl = pd.read_csv(intl_sales_path, low_memory=False)

# Step 26: Add a 'Channel' marker column to both dataframes before merging
# This ensures we can filter 'Local' vs 'International' easily in Power BI
df_amazon['Market_Channel'] = 'Local_Amazon'
df_intl['Market_Channel'] = 'International'

# Step 27: Align and concatenate both DataFrames vertically (row-bind)
# axis=0 means we are stacking rows on top of rows
df_master = pd.concat([df_amazon, df_intl], axis=0, ignore_index=True)

# Step 28: Export the final cleaned Master Table to your data folder
output_path = '../data/Cleaned_Master_Sales.csv'
df_master.to_csv(output_path, index=False)

# Step 29: Print execution success message and the new master shape
print("¡Data Consolidation Successful!")
print(f"Final Master Dataset Shape (Rows, Columns): {df_master.shape}")

¡Data Consolidation Successful!
Final Master Dataset Shape (Rows, Columns): (166407, 30)


## Step 7: Final Data Audit & Export

### Summary of Accomplishments:
* **Data Integration:** Successfully consolidated local Amazon logs and international sales datasets into a single master structure, scaling the dataset to **166,407 records**.
* **Financial Integrity:** Fixed over 7,500 missing values in the `Amount` and `currency` columns, standardizing canceled orders to `0.0` to avoid misleading KPI targets.
* **Temporal and Categorical Fixes:** Standardized the `Date` column into a native `datetime` object (spanning from March 31 to June 29, 2022) and corrected geographic text to Title Case.

### Next Stage:
The final dataset has been exported to `../data/Cleaned_Master_Sales.csv`. The project is now ready for **Phase 2: SQL Database Injection and Advanced Querying**, where we will define schemas, primary keys, and load this table into our relational database.

In [27]:
!pip install sqlalchemy pyodbc

In [28]:
import sqlalchemy as sa
from sqlalchemy import create_engine
import urllib

# Step 30: Configure connection string parameters for local SQL Server
# 'TrustServerCertificate=yes' avoids SSL errors in local environments
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=ECommerceSalesDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# Step 31: Initialize the SQLAlchemy Engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
print("¡SQL Server Connection Engine successfully initialized!")

¡SQL Server Connection Engine successfully initialized!


In [29]:
# --- DIMENSIONS INJECTION WITH ASSURED DEFAULT RECORDS ---

# Step 32: Extract and populate Dim_Status safely
status_cols = ['Status', 'Courier Status', 'Fulfilment', 'Market_Channel']
status_data = df_master[status_cols].copy()
status_data.columns = ['Status', 'Courier_Status', 'Fulfilment', 'Market_Channel']

status_data['Status'] = status_data['Status'].fillna('Unknown').astype(str).str.strip()
status_data['Courier_Status'] = status_data['Courier_Status'].fillna('Unknown').astype(str).str.strip()
status_data['Fulfilment'] = status_data['Fulfilment'].fillna('Unknown').astype(str).str.strip()
status_data['Market_Channel'] = status_data['Market_Channel'].fillna('Unknown').astype(str).str.strip()

# Add a guaranteed fallback default record row
default_status = pd.DataFrame([['Unknown', 'Unknown', 'Unknown', 'Unknown']], columns=status_data.columns)
status_data = pd.concat([default_status, status_data]).drop_duplicates().reset_index(drop=True)

status_data.to_sql('Dim_Status', con=engine, if_exists='append', index=False)
print("Dim_Status successfully loaded into production!")


# Step 33: Extract and populate Dim_Products safely
products_data = df_master[['SKU', 'ASIN', 'Style', 'Category', 'Size']].copy()
products_data['SKU'] = products_data['SKU'].fillna('Unknown').astype(str).str.strip()
products_data['ASIN'] = products_data['ASIN'].fillna('Unknown').astype(str).str.strip()
products_data['Style'] = products_data['Style'].fillna('Unknown').astype(str).str.strip()
products_data['Category'] = products_data['Category'].fillna('Unknown').astype(str).str.strip()
products_data['Size'] = products_data['Size'].fillna('Unknown').astype(str).str.strip()

# Add a guaranteed fallback default record row
default_product = pd.DataFrame([['Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown']], columns=products_data.columns)
products_data = pd.concat([default_product, products_data]).drop_duplicates().reset_index(drop=True)

products_data.to_sql('Dim_Products', con=engine, if_exists='append', index=False)
print("Dim_Products successfully loaded into production!")


# Step 34: Extract and populate Dim_Geography safely
geo_cols = [col for col in df_master.columns if 'city' in col.lower() or 'state' in col.lower() or 'postal' in col.lower()]
geo_data = df_master[geo_cols].copy()
geo_data.columns = ['Ship_City', 'Ship_State', 'Ship_Postal_Code']

geo_data['Ship_City'] = geo_data['Ship_City'].fillna('Unknown').astype(str).str.strip()
geo_data['Ship_State'] = geo_data['Ship_State'].fillna('Unknown').astype(str).str.strip()
geo_data['Ship_Postal_Code'] = geo_data['Ship_Postal_Code'].fillna('Unknown').astype(str).str.strip()

# Add a guaranteed fallback default record row
default_geo = pd.DataFrame([['Unknown', 'Unknown', 'Unknown']], columns=geo_data.columns)
geo_data = pd.concat([default_geo, geo_data]).drop_duplicates().reset_index(drop=True)

geo_data.to_sql('Dim_Geography', con=engine, if_exists='append', index=False)
print("Dim_Geography successfully loaded into production!")

Dim_Status successfully loaded into production!
Dim_Products successfully loaded into production!
Dim_Geography successfully loaded into production!


In [30]:
# --- DYNAMIC FACT TABLE MAPPING AND STREAMING WITH ANOMALY FILTER ---

# Step 35: Pull generated surrogate keys from database engine
df_status_sql = pd.read_sql("SELECT Status_ID, Status, Courier_Status, Fulfilment, Market_Channel FROM Dim_Status", con=engine)
df_products_sql = pd.read_sql("SELECT Product_ID, SKU, ASIN, Style, Category, Size FROM Dim_Products", con=engine)
df_geo_sql = pd.read_sql("SELECT Location_ID, Ship_City, Ship_State, Ship_Postal_Code FROM Dim_Geography", con=engine)

# Strip string white spaces on relational mapping frames to secure exact join executions
for df_sql in [df_status_sql, df_products_sql, df_geo_sql]:
    for col in df_sql.select_dtypes(include=['object']).columns:
        df_sql[col] = df_sql[col].astype(str).str.strip()

# DYNAMIC SAFEGUARD: Find the real auto-assigned IDs for the 'Unknown' records in the database
fallback_status_id = int(df_status_sql.loc[df_status_sql['Status'] == 'Unknown', 'Status_ID'].iloc[0])
fallback_product_id = int(df_products_sql.loc[df_products_sql['SKU'] == 'Unknown', 'Product_ID'].iloc[0])
fallback_location_id = int(df_geo_sql.loc[df_geo_sql['Ship_City'] == 'Unknown', 'Location_ID'].iloc[0])

# Step 36: Clean and prepare working dataset copy
df_master_mapping = df_master.copy()
df_master_mapping.rename(columns={
    'Courier Status': 'Courier_Status',
    'ship-city': 'Ship_City',
    'ship-state': 'Ship_State',
    'ship-postal-code': 'Ship_Postal_Code'
}, inplace=True)

# Homologate text attributes using exact string normalization rules
df_master_mapping['Status'] = df_master_mapping['Status'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Courier_Status'] = df_master_mapping['Courier_Status'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Fulfilment'] = df_master_mapping['Fulfilment'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Market_Channel'] = df_master_mapping['Market_Channel'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['SKU'] = df_master_mapping['SKU'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['ASIN'] = df_master_mapping['ASIN'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Style'] = df_master_mapping['Style'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Category'] = df_master_mapping['Category'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Size'] = df_master_mapping['Size'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Ship_City'] = df_master_mapping['Ship_City'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Ship_State'] = df_master_mapping['Ship_State'].fillna('Unknown').astype(str).str.strip()
df_master_mapping['Ship_Postal_Code'] = df_master_mapping['Ship_Postal_Code'].fillna('Unknown').astype(str).str.strip()

# Step 37: Execute relational joins via merges
df_fact = pd.merge(df_master_mapping, df_status_sql, on=['Status', 'Courier_Status', 'Fulfilment', 'Market_Channel'], how='left')
df_fact = pd.merge(df_fact, df_products_sql, on=['SKU', 'ASIN', 'Style', 'Category', 'Size'], how='left')
df_fact = pd.merge(df_fact, df_geo_sql, on=['Ship_City', 'Ship_State', 'Ship_Postal_Code'], how='left')

# CRITICAL SAFEGUARD: Coerce any unmapped records using the actual dynamic database IDs discovered
df_fact['Status_ID'] = df_fact['Status_ID'].fillna(fallback_status_id).astype(int)
df_fact['Product_ID'] = df_fact['Product_ID'].fillna(fallback_product_id).astype(int)
df_fact['Location_ID'] = df_fact['Location_ID'].fillna(fallback_location_id).astype(int)

# Step 38 & 39: Isolate schema layout and perform structural mapping
fact_columns = ['Order ID', 'Date', 'Qty', 'Amount', 'Status_ID', 'Product_ID', 'Location_ID']
df_fact = df_fact[fact_columns]
df_fact.rename(columns={'Order ID': 'Order_ID'}, inplace=True)

# DATA CLEANING VITAL FIX: Drop any rows where Order_ID is missing or null to avoid SQL constraint rejections
df_fact = df_fact.dropna(subset=['Order_ID'])
df_fact = df_fact[df_fact['Order_ID'].astype(str).str.lower() != 'nan']

# Step 40: Fast bulk-insert execution into SQL Server using streaming chunks
print("Uploading rows to Fact_Sales... Please hold on.")
df_fact.to_sql('Fact_Sales', con=engine, if_exists='append', index=False, chunksize=10000)

print(f"Data Pipeline Complete! Successfully loaded {len(df_fact)} records into Fact_Sales.")

C:\Users\52444\AppData\Local\Temp\ipykernel_1972\3664702216.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_sql.select_dtypes(include=['object']).columns:
C:\Users\52444\AppData\Local\Temp\ipykernel_1972\3664702216.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_

Uploading rows to Fact_Sales... Please hold on.
Data Pipeline Complete! Successfully loaded 128975 records into Fact_Sales.


In [31]:
# --- DATA QUALITY ASSURANCE & SANITY CHECKS ---

import pandas as pd

print("Starting Data Quality Assurance checks...")

# 1. Record Count Validation
# Fetch total rows directly from the production Fact table
sql_fact_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM Fact_Sales", con=engine).iloc[0]['total_rows']

# Filter out rows from df_master where 'Order ID' is missing (matching our pipeline cleaning logic)
df_master_clean = df_master.dropna(subset=['Order ID'])
df_master_clean = df_master_clean[df_master_clean['Order ID'].astype(str).str.lower() != 'nan']
pandas_source_count = len(df_master_clean)

print(f" -> Source Rows (Pandas Cleaned): {pandas_source_count}")
print(f" -> Destination Rows (SQL Server): {sql_fact_count}")

if pandas_source_count == sql_fact_count:
    print(" [SUCCESS] Record counts match perfectly!")
else:
    print(" [WARNING] Record counts mismatch. Please verify dropped or duplicate entries.")

# 2. Financial Metrics Validation
# Calculate total revenue from source versus destination to ensure no numeric data loss
pandas_total_amount = df_master_clean['Amount'].sum()
sql_total_amount = pd.read_sql("SELECT SUM(Amount) AS total_amount FROM Fact_Sales", con=engine).iloc[0]['total_amount']

print(f" -> Source Total Revenue: ${pandas_total_amount:,.2f}")
print(f" -> Destination Total Revenue: ${sql_total_amount:,.2f}")

# Using a small delta tolerance for floating-point comparison
if abs(pandas_total_amount - sql_total_amount) < 0.01:
    print(" [SUCCESS] Financial reconciliation complete! Dollar amounts match perfectly.")
else:
    print(f" [WARNING] Revenue mismatch detected. Difference: ${abs(pandas_total_amount - sql_total_amount):,.2f}")

Starting Data Quality Assurance checks...
 -> Source Rows (Pandas Cleaned): 128975
 -> Destination Rows (SQL Server): 128975
 [SUCCESS] Record counts match perfectly!
 -> Source Total Revenue: $78,592,678.30
 -> Destination Total Revenue: $78,592,678.30
 [SUCCESS] Financial reconciliation complete! Dollar amounts match perfectly.


In [32]:
# --- DYNAMIC FACT TABLE MAPPING AND STREAMING WITH ANOMALY FILTER ---

import pandas as pd
from sqlalchemy import create_engine

# Step 35: Pull generated surrogate keys from database engine
df_status_sql = pd.read_sql("SELECT Status_ID, Status, Courier_Status, Fulfilment, Market_Channel FROM Dim_Status", con=engine)
df_products_sql = pd.read_sql("SELECT Product_ID, SKU, ASIN, Style, Category, Size FROM Dim_Products", con=engine)
df_geo_sql = pd.read_sql("SELECT Location_ID, Ship_City, Ship_State, Ship_Postal_Code FROM Dim_Geography", con=engine)

# Strip string white spaces on relational mapping frames to secure exact join executions
for df_sql in [df_status_sql, df_products_sql, df_geo_sql]:
    for col in df_sql.select_dtypes(include=['object']).columns:
        df_sql[col] = df_sql[col].astype(str).str.strip()

# DYNAMIC SAFEGUARD: Find the real auto-assigned IDs for the 'Unknown' records in the database
fallback_status_id = int(df_status_sql.loc[df_status_sql['Status'] == 'Unknown', 'Status_ID'].iloc[0])
fallback_product_id = int(df_products_sql.loc[df_products_sql['SKU'] == 'Unknown', 'Product_ID'].iloc[0])
fallback_location_id = int(df_geo_sql.loc[df_geo_sql['Ship_City'] == 'Unknown', 'Location_ID'].iloc[0])

# Step 36: Clean and prepare working dataset copy
df_master_mapping = df_master.copy()
df_master_mapping.rename(columns={
    'Courier Status': 'Courier_Status',
    'ship-city': 'Ship_City',
    'ship-state': 'Ship_State',
    'ship-postal-code': 'Ship_Postal_Code'
}, inplace=True)

# Homologate text attributes using exact string normalization rules
for col in ['Status', 'Courier_Status', 'Fulfilment', 'Market_Channel', 'SKU', 'ASIN', 'Style', 'Category', 'Size', 'Ship_City', 'Ship_State', 'Ship_Postal_Code']:
    df_master_mapping[col] = df_master_mapping[col].fillna('Unknown').astype(str).str.strip()

# Step 37: Execute relational joins via merges
df_fact = pd.merge(df_master_mapping, df_status_sql, on=['Status', 'Courier_Status', 'Fulfilment', 'Market_Channel'], how='left')
df_fact = pd.merge(df_fact, df_products_sql, on=['SKU', 'ASIN', 'Style', 'Category', 'Size'], how='left')
df_fact = pd.merge(df_fact, df_geo_sql, on=['Ship_City', 'Ship_State', 'Ship_Postal_Code'], how='left')

# CRITICAL SAFEGUARD: Coerce any unmapped records using the actual dynamic database IDs discovered
df_fact['Status_ID'] = df_fact['Status_ID'].fillna(fallback_status_id).astype(int)
df_fact['Product_ID'] = df_fact['Product_ID'].fillna(fallback_product_id).astype(int)
df_fact['Location_ID'] = df_fact['Location_ID'].fillna(fallback_location_id).astype(int)

# Step 38 & 39: Isolate schema layout and perform structural mapping
fact_columns = ['Order ID', 'Date', 'Qty', 'Amount', 'Status_ID', 'Product_ID', 'Location_ID']
df_fact = df_fact[fact_columns]
df_fact.rename(columns={'Order ID': 'Order_ID'}, inplace=True)

# DATA CLEANING VITAL FIX: Drop any rows where Order_ID is missing or null to avoid SQL constraint rejections
df_fact = df_fact.dropna(subset=['Order_ID'])
df_fact = df_fact[df_fact['Order_ID'].astype(str).str.lower() != 'nan']

# Step 40: Fast bulk-insert execution into SQL Server using streaming chunks
print("Uploading rows to Fact_Sales... Please hold on.")
df_fact.to_sql('Fact_Sales', con=engine, if_exists='append', index=False, chunksize=10000)
print(f"Data Pipeline Complete! Successfully loaded {len(df_fact)} records into Fact_Sales.")

C:\Users\52444\AppData\Local\Temp\ipykernel_1972\3685909941.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_sql.select_dtypes(include=['object']).columns:
C:\Users\52444\AppData\Local\Temp\ipykernel_1972\3685909941.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_

Uploading rows to Fact_Sales... Please hold on.
Data Pipeline Complete! Successfully loaded 128975 records into Fact_Sales.


In [33]:
# --- DATA QUALITY ASSURANCE & RECONCILIATION ---

sql_fact_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM Fact_Sales", con=engine).iloc[0]['total_rows']
df_master_clean = df_master.dropna(subset=['Order ID'])
df_master_clean = df_master_clean[df_master_clean['Order ID'].astype(str).str.lower() != 'nan']
pandas_source_count = len(df_master_clean)

print(f"Source Rows (Cleaned): {pandas_source_count} | Database Target Rows: {sql_fact_count}")
if pandas_source_count == sql_fact_count:
    print("[QA PASSED] Record counts match perfectly!")

# Financial Validation Checks
pandas_revenue = df_master_clean['Amount'].sum()
sql_revenue = pd.read_sql("SELECT SUM(Amount) AS total_amount FROM Fact_Sales", con=engine).iloc[0]['total_amount']
print(f"Source Revenue: ${pandas_revenue:,.2f} | Database Revenue: ${sql_revenue:,.2f}")
if abs(pandas_revenue - sql_revenue) < 0.01:
    print("[QA PASSED] Financial reconciliation complete!")

Source Rows (Cleaned): 128975 | Database Target Rows: 257950
Source Revenue: $78,592,678.30 | Database Revenue: $157,185,356.60
